# 02 · Amostragem do gabarito

Sorteia os 800 comentários para anotação manual (estratificados por nota e atraso), separa validação/teste e gera o arquivo do segundo anotador.

**Rode uma única vez.** Depois de anotar, o arquivo vai para o git.

In [ ]:
# Configuração: no Colab, clona o repositório e instala o pacote `absa`
import sys
if "google.colab" in sys.modules:
    !git clone -q https://github.com/vnevesl/pln-absa-olist.git 2>/dev/null || git -C pln-absa-olist pull -q
    %cd pln-absa-olist
    !pip -q install -e "."
    # Guarda cache do LLM e modelos no Drive para sobreviver à queda da sessão
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/pln-absa-olist/llm_cache /content/drive/MyDrive/pln-absa-olist/models
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/llm_cache data/llm_cache
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/models models

In [ ]:
from absa.config import ANNOTATED, LABEL_COLUMNS
from absa.data import load_reviews
from absa.sampling import double_annotation_subset, sample_gold

df = load_reviews()
gold = sample_gold(df)
print(len(gold))
print(gold.groupby("nota").atrasado.agg(["size", "mean"]).round(2))

In [ ]:
ANNOTATED.mkdir(parents=True, exist_ok=True)
# A nota fica fora do arquivo de anotação para não influenciar quem anota
cols = ["ordem_anotacao", "review_id", "texto"]
para_anotar = gold[cols].copy()
for c in LABEL_COLUMNS + ["obs"]:
    para_anotar[c] = ""
destino = ANNOTATED / "gabarito_para_anotar.csv"
assert not destino.exists(), "Já existe: não sobrescreva anotações feitas!"
para_anotar.to_csv(destino, index=False)

segundo = double_annotation_subset(para_anotar)
segundo.to_csv(ANNOTATED / "segundo_anotador.csv", index=False)
gold.to_csv(ANNOTATED / "gabarito_metadados.csv", index=False)
print("arquivos salvos em", ANNOTATED)

Abra `gabarito_para_anotar.csv` no Google Planilhas ou Excel, siga `anotacao/guia_de_anotacao.md` e salve como `gabarito_anotado.csv` na mesma pasta.